In [ ]:
import sys
import os
import shutil
import pandas as pd
import numpy as np
import scipy.stats
import json

In [ ]:
# Get all json files containing the results to analzyse EXPERIMENT 2

# copy all 10 iteration results into a new folder

base_dir = "./NanoDesiger/iterative_process_DiffAb_denovo"
out_dir = "./NanoDesiger/iterative_process_DiffAb_denovo_results"
csv_name = "NanoDesigner_DIFFAB_3CDRs_de_novo.csv" # define name of output csv file
list_sub = os.listdir(base_dir)
list_sub = [element for element in list_sub if len(element) == 4]

for subdir in list_sub:
    subdir_path = os.path.join(base_dir, subdir)
    
    if os.path.isdir(subdir_path):
        target_subdir = os.path.join(out_dir, subdir)
        os.makedirs(target_subdir, exist_ok=True)  
        
        # Loop through all files in the current subdirectory
        for filename in os.listdir(subdir_path):
            # Check if the filename contains "best_candidates_iter_"
            if "best_candidates_iter_" in filename and filename.endswith(".json"):
                source_file_path = os.path.join(subdir_path, filename)
                destination_file_path = os.path.join(target_subdir, filename)
                shutil.copy(source_file_path, destination_file_path)
                print(f"Copied {source_file_path} to {destination_file_path}")

In [ ]:


# Define the function to calculate the margin of error (half confidence interval)
def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data) 
    n = len(a) 
    m = np.mean(a)  
    se = scipy.stats.sem(a) 
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n - 1) 
    return h  


def list_main_directories(main_folder):
    return [os.path.join(main_folder, subfolder) for subfolder in os.listdir(main_folder) if os.path.isdir(os.path.join(main_folder, subfolder))]

def find_files_for_iterations(directories, iterations):
    print(directories)
    print(iterations)
    files_by_iteration = {}
    for directory in directories:
        protein_id = os.path.basename(directory)  # use subfolder name as identifier
        files_by_iteration[protein_id] = {}
        for iteration in iterations:
            iteration_files = []
            # Assumes files are named like "fold_X.json" where X is the iteration number
            for file_name in os.listdir(directory):
                # if f'fold_{iteration}' in file_name and file_name.endswith('.json'):
                if f'best_candidates_iter_{iteration}.json' in file_name and file_name.endswith('.json'):
                    iteration_files.append(os.path.join(directory, file_name))
            if iteration_files:
                files_by_iteration[protein_id][iteration] = iteration_files
    return files_by_iteration



def load_gt_dg_data(gt_dg_file_path):
    gt_dg_data = {}
    with open(gt_dg_file_path, 'r') as f:
        data = [json.loads(line) for line in f]
    for entry in data:
        key_1 = entry['pdb_data_path']
        key_2 = f"{entry['pdb'].split('_')[0]}_{entry['heavy_chain']}"
        gt_dg_data[key_2] = entry['FoldX_dG']
    
    return gt_dg_data

def compute_ddG(dG, gt_dG):
    if gt_dG is not None:
        return dG - gt_dG
    else:
        return None


def compile_data(files_by_iteration, gt_dg_data):
    data = []
    for protein_id, iteration_files in files_by_iteration.items():
        for iteration, files in iteration_files.items():
            for file_path in files:
                with open(file_path, 'r') as f:
                    json_data = [json.loads(line) for line in f]
                    for entry in json_data:
                        if 'FoldX_dG' in entry and 'ref_pdb' in entry and 'heavy_chain' in entry:
                            key_2 = f"{entry['pdb'].split('_')[0]}_{entry['heavy_chain']}"
                            gt_dg = gt_dg_data.get(key_2, None)
                            computed_ddG = compute_ddG(entry['FoldX_dG'], gt_dg)

                            # Use computed_ddG if FoldX_ddG is missing or not matching
                            final_ddG = entry.get('FoldX_ddG', computed_ddG)
                            if computed_ddG is not None and final_ddG != computed_ddG:
                                final_ddG = computed_ddG  # Override with computed one if it exists

                            entry_data = {
                                'ProteinID': protein_id,
                                'iteration': iteration,
                                'FoldX_dG': entry['FoldX_dG'],
                                'GT_dG': gt_dg,
                                'FoldX_ddG': final_ddG,  # Use final_ddG (either from file or computed)
                                'AAR H3': entry.get('AAR H3', None),
                                'AAR H2': entry.get('AAR H2', None),
                                'AAR H1': entry.get('AAR H1', None),
                                'RMSD': entry.get('RMSD(CA) aligned', None),
                                'RMSD_cdrh3': entry.get('RMSD(CA) CDRH3 aligned', None),
                                'RMSD_cdrh2': entry.get('RMSD(CA) CDRH2 aligned', None),
                                'RMSD_cdrh1': entry.get('RMSD(CA) CDRH1 aligned', None),
                                'TMscore': entry.get('TMscore', None),
                                'LDDT': entry.get('LDDT', None),
                                'DockQ': entry.get('DockQ', None),
                                'Num_Clashes': entry.get('final_num_clashes', None)
                            }
                            data.append(entry_data)
    return data



def remove_outliers_iqr(df, column):
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    
    # Calculate the Interquartile Range (IQR)
    IQR = Q3 - Q1
    
    # Define lower and upper bounds for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filter out outliers
    filtered_df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    
    return filtered_df


"""
FOR EXPERIMENT 2 RESULTS
"""

def create_summary_table(df):
    # Define the columns for which we need statistics and their boundaries
    metrics = ['AAR', 'RMSD', 'RMSD_cdrh3', 'TMscore', 'Lddt', 'DockQ', 'FoldX_dG', 'FoldX_ddG', 'Num_Clashes', 'iteration']
    
    boundaries = {
        'AAR': (0.0, 1.0),
        'RMSD': (0.0, None),
        'RMSD_cdrh3': (0.0, None),
        'TMscore': (0.0, 1.0),
        'Lddt': (0.0, 1.0),
        'DockQ': (0.0, None),
        'FoldX_dG': (None, None),
        'FoldX_ddG': (None, None),
        'Num_Clashes': (0.0, None)
    }

    # Aggregation for the main metrics
    agg_dict = {}
    for metric in metrics:
        agg_dict[f'{metric}_mean'] = (metric, 'mean')  # Calculate the mean
        agg_dict[f'{metric}_std'] = (metric, lambda x: x.std(ddof=1))  # Calculate the standard deviation
        # Calculate the margin of error (1.96 * SD / sqrt(n))
        agg_dict[f'{metric}_CI_margin'] = (metric, lambda x: 1.96 * x.std(ddof=1) / np.sqrt(len(x)))

    # Add success rate calculation (ddG < 0)
    agg_dict['success_rate'] = ('FoldX_ddG', lambda x: (x < 0).mean())  # Success rate as proportion
    agg_dict['n_samples'] = ('FoldX_ddG', 'count')  # Sample size for the success rate

    # Aggregate the data by iteration (grouped by iteration)
    summary = df.groupby('iteration').agg(**agg_dict)

    # Debugging: Print the first few rows of the aggregated data
    print(summary.head())  # Check if FoldX_dG and other metrics are aggregated correctly

    # Apply boundaries to the summary table
    for metric, (lower_bound, upper_bound) in boundaries.items():
        # Apply clipping to the mean
        if lower_bound is not None:
            summary[f'{metric}_mean'] = summary[f'{metric}_mean'].clip(lower=lower_bound)
        if upper_bound is not None:
            summary[f'{metric}_mean'] = summary[f'{metric}_mean'].clip(upper=upper_bound)

    # Calculate lower and upper bounds of the confidence intervals for all metrics
    for metric in metrics:
        summary[f'{metric}_CI_lower'] = summary[f'{metric}_mean'] - summary[f'{metric}_CI_margin']
        summary[f'{metric}_CI_upper'] = summary[f'{metric}_mean'] + summary[f'{metric}_CI_margin']

    # Calculate the 95% CI margin for the success rate (as a proportion)
    summary['success_rate_CI_margin'] = 1.96 * np.sqrt((summary['success_rate']) * (1 - summary['success_rate']) / summary['n_samples'])

    # Multiply success rate and CI margin by 100 to convert back to percentage
    summary['success_rate'] = summary['success_rate'] * 100
    summary['success_rate_CI_margin'] = summary['success_rate_CI_margin'] * 100

    # Round all values to 2 decimals
    summary = summary.round(4)

    # Reset index to get 'iteration' as a column
    summary = summary.reset_index()

    return summary


def create_summary_table(df):

    metrics = [
        'AAR H3', 'RMSD', 'RMSD_cdrh3','RMSD_cdrh2', 'RMSD_cdrh1', "AAR H2", "AAR H1",
        'TMscore', 'LDDT', 'DockQ', 'FoldX_dG', 'FoldX_ddG', 
        'Num_Clashes'
    ]

    boundaries = {
        'AAR H3': (0.0, 1.0),
        'AAR H2': (0.0, 1.0),
        'AAR H1': (0.0, 1.0),
        'RMSD': (0.0, None),
        'RMSD_cdrh3': (0.0, None),
        'RMSD_cdrh2': (0.0, None),
        'RMSD_cdrh1': (0.0, None),
        'TMscore': (0.0, 1.0),
        'LDDT': (0.0, 1.0),
        'DockQ': (0.0, None),
        'FoldX_dG': (None, None),
        'FoldX_ddG': (None, None),
        'Num_Clashes': (0.0, None)
    }

    # Aggregation for the main metrics
    agg_dict = {}
    for metric in metrics:
        agg_dict[f'{metric}_mean'] = (metric, 'mean')  # Calculate the mean
        agg_dict[f'{metric}_std'] = (metric, lambda x: x.std(ddof=1))  # Calculate the standard deviation
        # Calculate the margin of error (1.96 * SD / sqrt(n))
        agg_dict[f'{metric}_CI_margin'] = (metric, lambda x: 1.96 * x.std(ddof=1) / np.sqrt(len(x)))
        # Calculate the margin of error using mean_confidence_interval function
        agg_dict[f'{metric}_CI_margin_stats_library'] = (metric, lambda x: mean_confidence_interval(x))

    # Add success rate calculation (ddG < 0)
    agg_dict['success_rate'] = ('FoldX_ddG', lambda x: (x < 0).mean())  # Success rate as proportion
    agg_dict['n_samples'] = ('FoldX_ddG', 'count')  # Sample size for the success rate
    agg_dict['success_rate_CI_margin_stats_library'] = ('FoldX_ddG', lambda x: mean_confidence_interval((x < 0) * 100))  # CI for success rate

    # Aggregate the data by iteration (grouped by iteration)
    summary = df.groupby('iteration').agg(**agg_dict)

    # Debugging: Print the first few rows of the aggregated data
    print(summary.head())  # Check if FoldX_dG and other metrics are aggregated correctly

    # Apply boundaries to the summary table
    for metric, (lower_bound, upper_bound) in boundaries.items():
        # Apply clipping to the mean
        if lower_bound is not None:
            summary[f'{metric}_mean'] = summary[f'{metric}_mean'].clip(lower=lower_bound)
        if upper_bound is not None:
            summary[f'{metric}_mean'] = summary[f'{metric}_mean'].clip(upper=upper_bound)

    # Calculate lower and upper bounds of the confidence intervals for all metrics
    for metric in metrics:
        summary[f'{metric}_CI_lower'] = summary[f'{metric}_mean'] - summary[f'{metric}_CI_margin']
        summary[f'{metric}_CI_upper'] = summary[f'{metric}_mean'] + summary[f'{metric}_CI_margin']

    # Calculate the 95% CI margin for the success rate (as a proportion)
    summary['success_rate_CI_margin'] = 1.96 * np.sqrt((summary['success_rate']) * (1 - summary['success_rate']) / summary['n_samples'])

    # Multiply success rate and CI margin by 100 to convert back to percentage
    summary['success_rate'] = summary['success_rate'] * 100
    summary['success_rate_CI_margin'] = summary['success_rate_CI_margin'] * 100

    # Round all values to 2 decimals
    summary = summary.round(4)

    # Reset index to get 'iteration' as a column
    summary = summary.reset_index()

    return summary


base_dir = os.path.dirname(os.getcwd())
gt_dg_file_path = os.path.join(os.path.join(base_dir,"functionalities"),"Nanobody_dataset_with_FoldX_dG_data_july_2024.json")
iterations = range(1, 11)

folder_path = out_dir

directories = list_main_directories(folder_path)
gt_dg_data = load_gt_dg_data(gt_dg_file_path)
files_by_iteration = find_files_for_iterations(directories, iterations)
data = compile_data(files_by_iteration, gt_dg_data)
df = pd.DataFrame(data)
summary_table = create_summary_table(df)
print(summary_table)
summary_table.to_csv(csv_name, index=False)

In [ ]:
"""
FOR EXPERIMENT 1 RESULTS
"""

# Function to load the ground truth dG data
def load_gt_dg_data(gt_dg_file_path):
    gt_dg_data = {}
    with open(gt_dg_file_path, 'r') as f:
        data = [json.loads(line) for line in f]
    for entry in data:
        key_1 = entry['pdb_data_path']
        key_2 = f"{entry['pdb'].split('_')[0]}_{entry['heavy_chain']}"
        gt_dg_data[key_2] = entry['FoldX_dG']
    return gt_dg_data

# Function to compute ddG
def compute_ddG(dG, gt_dG):
    if gt_dG is not None:
        return dG - gt_dG
    else:
        return None

# Function to compile data from JSON files
def compile_data(subdir, gt_dg_data):
    data = []
    for file_name in os.listdir(subdir):
        if "metrics_fold" in file_name and file_name.endswith(".json"):
            file_path = os.path.join(subdir, file_name)
            with open(file_path, 'r') as f:
                json_data = [json.loads(line) for line in f]
                for entry in json_data:
                    if 'FoldX_dG' in entry and 'pdb' in entry and 'heavy_chain' in entry:
                        key_2 = f"{entry['pdb'].split('_')[0]}_{entry['heavy_chain']}"
                        gt_dg = gt_dg_data.get(key_2, None)
                        computed_ddG = compute_ddG(entry['FoldX_dG'], gt_dg)

                        # Use computed_ddG if FoldX_ddG is missing or not matching
                        final_ddG = entry.get('FoldX_ddG', computed_ddG)
                        if computed_ddG is not None and final_ddG != computed_ddG:
                            final_ddG = computed_ddG  # Override with computed one if it exists

                        entry_data = {
                            'Subdir': os.path.basename(subdir),
                            'Fold': file_path.split('_')[-1].split('.')[0],
                            'FoldX_dG': entry['FoldX_dG'],
                            'GT_dG': gt_dg,
                            'FoldX_ddG': final_ddG,
                            'AAR': entry.get('AAR H3', None),
                            'RMSD': entry.get('RMSD(CA) aligned', None),
                            'RMSD_cdrh3': entry.get('RMSD(CA) CDRH3 aligned', None),
                            'TMscore': entry.get('TMscore', None),
                            'Lddt': entry.get('LDDT', None),
                            'DockQ': entry.get('DockQ', None),
                            'Num_Clashes': entry.get('inference_clashes', None)
                        }
                        data.append(entry_data)
    return data


# Function to create a summary table
def create_summary_table(df):
    metrics = ['AAR', 'RMSD', 'RMSD_cdrh3', 'TMscore', 'Lddt', 'DockQ', 'FoldX_dG', 'FoldX_ddG', 'Num_Clashes']

    boundaries = {
        'AAR': (0.0, 1.0),
        'RMSD': (0.0, None),
        'RMSD_cdrh3': (0.0, None),
        'TMscore': (0.0, 1.0),
        'Lddt': (0.0, 1.0),
        'DockQ': (0.0, None),
        'FoldX_dG': (None, None),
        'FoldX_ddG': (None, None),
        'Num_Clashes': (0.0, None)
    }

    # Aggregation for the main metrics by fold first
    fold_agg = {}
    for metric in metrics:
        fold_agg[metric] = ['mean', 'std']  # Calculate the mean and standard deviation for each fold
    
    # Group the data by Fold and Subdir (fold-level aggregation)
    fold_summary = df.groupby(['Subdir', 'Fold']).agg(fold_agg).reset_index()

    # Flatten multi-level columns
    fold_summary.columns = ['_'.join(col).rstrip('_') for col in fold_summary.columns]

    # Manually calculate success rate (proportion of negative ddG) and sample count per fold
    fold_summary['success_rate'] = df.groupby(['Subdir', 'Fold'])['FoldX_ddG'].apply(lambda x: (x < 0).mean()).values
    fold_summary['n_samples'] = df.groupby(['Subdir', 'Fold'])['FoldX_ddG'].apply(len).values

    # Now aggregate across subfolders (across folds)
    subfolder_agg = {}
    for metric in metrics:
        subfolder_agg[f'{metric}_mean'] = (f'{metric}_mean', 'mean')  # Mean of the fold means
        
        # Instead of using a static margin, we now use mean_confidence_interval
        subfolder_agg[f'{metric}_CI_margin'] = (f'{metric}_mean', lambda x: mean_confidence_interval(x))  # Margin of error using the confidence interval

    # Aggregation for overall success rate across subfolders
    subfolder_agg['success_rate_mean'] = ('success_rate', 'mean')
    subfolder_agg['success_rate_std'] = ('success_rate', 'std')
    subfolder_agg['success_rate_CI_margin'] = ('success_rate', lambda x: mean_confidence_interval(x))

    # Group by Subdir
    summary = fold_summary.groupby('Subdir').agg(**subfolder_agg).reset_index()

    # Apply boundaries to the summary table
    for metric, (lower_bound, upper_bound) in boundaries.items():
        if lower_bound is not None:
            summary[f'{metric}_mean'] = summary[f'{metric}_mean'].clip(lower=lower_bound)
        if upper_bound is not None:
            summary[f'{metric}_mean'] = summary[f'{metric}_mean'].clip(upper=upper_bound)

    return summary


def create_summary_table(df):
    metrics = ['AAR', 'RMSD', 'RMSD_cdrh3', 'TMscore', 'Lddt', 'DockQ', 'FoldX_dG', 'FoldX_ddG', 'Num_Clashes']

    boundaries = {
        'AAR': (0.0, 1.0),
        'RMSD': (0.0, None),
        'RMSD_cdrh3': (0.0, None),
        'TMscore': (0.0, 1.0),
        'Lddt': (0.0, 1.0),
        'DockQ': (0.0, None),
        'FoldX_dG': (None, None),
        'FoldX_ddG': (None, None),
        'Num_Clashes': (0.0, None)
    }

    # Aggregation for the main metrics by fold first
    fold_agg = {}
    for metric in metrics:
        fold_agg[metric] = ['mean', 'std']  # Calculate the mean and standard deviation for each fold
    
    # Group the data by Fold and Subdir (fold-level aggregation)
    fold_summary = df.groupby(['Subdir', 'Fold']).agg(fold_agg).reset_index()

    # Flatten multi-level columns
    fold_summary.columns = ['_'.join(col).rstrip('_') for col in fold_summary.columns]

    # Manually calculate success rate (proportion of negative ddG) and sample count per fold
    fold_summary['success_rate'] = df.groupby(['Subdir', 'Fold'])['FoldX_ddG'].apply(lambda x: (x < 0).mean()).values
    fold_summary['n_samples'] = df.groupby(['Subdir', 'Fold'])['FoldX_ddG'].apply(len).values

    # Now aggregate across subfolders (across folds)
    subfolder_agg = {}
    for metric in metrics:
        subfolder_agg[f'{metric}_mean'] = (f'{metric}_mean', 'mean')  # Mean of the fold means
        
        # CI using standard deviation formula (existing method)
        subfolder_agg[f'{metric}_CI_margin'] = (f'{metric}_mean', lambda x: 1.96 * x.std(ddof=1) / np.sqrt(len(x)))

        # CI using mean_confidence_interval (new method)
        subfolder_agg[f'{metric}_CI_margin_stats_library'] = (f'{metric}_mean', lambda x: mean_confidence_interval(x))  # Margin of error using the confidence interval

    # Aggregation for overall success rate across subfolders
    subfolder_agg['success_rate_mean'] = ('success_rate', 'mean')
    subfolder_agg['success_rate_std'] = ('success_rate', 'std')
    
    # CI for success rate using standard deviation method
    subfolder_agg['success_rate_CI_margin'] = ('success_rate', lambda x: 1.96 * x.std(ddof=1) / np.sqrt(len(x)))

    # CI for success rate using mean_confidence_interval
    subfolder_agg['success_rate_CI_margin_stats_library'] = ('success_rate', lambda x: mean_confidence_interval(x * 100))  # Convert success rate to percentage for CI calculation

    # Group by Subdir
    summary = fold_summary.groupby('Subdir').agg(**subfolder_agg).reset_index()

    # Apply boundaries to the summary table
    for metric, (lower_bound, upper_bound) in boundaries.items():
        if lower_bound is not None:
            summary[f'{metric}_mean'] = summary[f'{metric}_mean'].clip(lower=lower_bound)
        if upper_bound is not None:
            summary[f'{metric}_mean'] = summary[f'{metric}_mean'].clip(upper=upper_bound)

    return summary

def aggregate_by_subdir_and_fold(df, output_csv_path):
    # Group by both "Subdir" and "Fold"
    grouped = df.groupby(['Subdir', 'Fold'])
    
    # Calculate the mean and standard deviation for each group
    mean_df = grouped.mean().reset_index()
    std_df = grouped.std().reset_index()

    # Merge the mean and std dataframes side by side, with appropriate suffixes
    result_df = pd.merge(mean_df, std_df, on=['Subdir', 'Fold'], suffixes=('_mean', '_std'))

    # Save the result to a CSV file
    result_df.to_csv(output_csv_path, index=False)

    return result_df

import scipy.stats

# Define the function to calculate the margin of error (half confidence interval)
def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)  # Convert the data to a NumPy array and ensure it's in float format
    n = len(a)  # Number of samples
    m = np.mean(a)  # Calculate the mean of the data
    se = scipy.stats.sem(a)  # Standard error of the mean
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n - 1)  # Compute the margin of error
    return h  # Return the margin of error (half the confidence interval)

Create a folder that has this structure

    """

    assssment_tools_per_fold/ # your args.out_folder
    └── Adesigner_Nano_Ag/
    └── Adesigner_Nano_cdrh3/
    └── Adesigner_Nano_Ab_Ag/
    └── Adesigner_Nano_Ab_cdrh3/
    └── DiffAb_Nano_Ag/
    ...
    └── dyMEAN_Nano_Ab_cdrh3/
        ├── metrics_fold_0.json
        ├── metrics_fold_1.json
        ...
        └── metrics_fold_9.json
    
    """

In [ ]:

# Paths to the main folder and the ground truth dG file
main_folder = "./assssment_tools_per_fold"

base_dir = os.path.dirname(os.getcwd())
gt_dg_file_path = os.path.join(os.path.join(base_dir,"functionalities"),"Nanobody_dataset_with_FoldX_dG_data_july_2024.json")
subdirs = [os.path.join(main_folder, d) for d in os.listdir(main_folder) if os.path.isdir(os.path.join(main_folder, d))]
gt_dg_data = load_gt_dg_data(gt_dg_file_path)

all_data = []
for subdir in subdirs:
    data = compile_data(subdir, gt_dg_data)
    all_data.extend(data)

# Load everything into a DataFrame
df = pd.DataFrame(all_data)

print(df)

output_csv = os.path.join(main_folder, 'assessment_tools_paper_per_fold.csv')
result_df = aggregate_by_subdir_and_fold(df, output_csv)

# Create a summary table
summary_table = create_summary_table(df)

print(summary_table)

# Save the summary as CSV
summary_table.to_csv(os.path.join(main_folder, 'assessment_tools_paper_oct2024_4.csv'), index=False)